# DocAuditor — Encoder-Only Document Intelligence

**Preeyas Tumulu · Generative AI · Unit Test 01**

**Five heads, one encoder family, zero hallucinations.**

A document-review assistant for legal, HR and compliance teams built **entirely from
BERT-family encoder models**. No generative model is used anywhere in this notebook.

## Why encoder-only matters here

A compliance officer cannot act on an answer they cannot verify. Every output in this
notebook is one of three things:

| Output type | What it is | Why it can be trusted |
|---|---|---|
| A **span** | character offsets into the source | you can point at it in the original |
| A **probability** | softmax over a fixed label set | it is calibrated and comparable |
| A **distance** | cosine similarity between embeddings | it is reproducible and inspectable |

None of these can be fabricated. An encoder has no decoder — there is no mechanism by
which it can invent a clause that isn't there. That is the entire argument for this
architecture in a regulated setting.

## Features

| # | Feature | Model | Head |
|---|---|---|---|
| 1 | PII Scanner | `iiiorg/piiranha-v1-detect-personal-information` | token classification |
| 2 | Document Q&A | `all-MiniLM-L6-v2` + `deepset/roberta-base-squad2` | retrieval + span QA |
| 3 | Ambiguity Meter | `bert-base-uncased` + `nli-deberta-v3-base` | MLM entropy + NLI |
| 4 | Contradiction Detector | `all-MiniLM-L6-v2` + `nli-deberta-v3-base` | bi-encoder filter + NLI |
| 5 | Document Auto-Router | `nli-deberta-v3-base` | zero-shot classification |

Features 1–3 are required by the brief; 4 and 5 are the free-choice features.

## Requirement mapping

| Brief requirement | Feature | App tab | Model |
|---|---|---|---|
| Scan the document for PII | PII Scanner | PII Scanner | `piiranha-v1` (mDeBERTa-v3) |
| Ask plain-English questions | Document Q&A | Ask the Document | `all-MiniLM-L6-v2` + `roberta-base-squad2` |
| Test ambiguous language | Ambiguity Meter | Ambiguity Meter | `bert-base-uncased` + `nli-deberta-v3-base` |
| Feature of choice #1 | Contradiction Detector | Contradictions | `all-MiniLM-L6-v2` + `nli-deberta-v3-base` |
| Feature of choice #2 | Document Auto-Router | Router | `nli-deberta-v3-base` |

**Input formats.** TXT, PDF and DOCX are all accepted (see `extract_text` below).
DOCX tables are flattened row by row, since contract schedules and HR forms keep
much of their sensitive detail in tables.

## Are these all "BERT"?

Yes. RoBERTa is BERT with improved pretraining (no NSP, dynamic masking). DeBERTa-v3 is
BERT with disentangled attention. MiniLM is a distilled BERT-family sentence encoder.
All are bidirectional transformer **encoders** — same architecture family, same
`[CLS]`/`[SEP]` tokenisation, same masked-language-model pretraining objective.

---
## 0. Setup

In [1]:
# !pip install torch transformers sentence-transformers pandas scikit-learn

import json, math, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Device: cuda
GPU:    NVIDIA GeForce RTX 3050 Laptop GPU
VRAM:   4.3 GB


### Model registry

Every model identifier lives in one dict. Two reasons: it documents in one place that
nothing here is generative, and swapping the entire stack to vanilla `bert-base-*`
checkpoints is a one-cell edit if the "BERT only" constraint is read strictly.

Models are loaded **lazily** — on a 4 GB GPU the full stack does not comfortably
coexist, so each is loaded on first use and cached.

In [2]:
MODELS = {
    "pii":       "iiiorg/piiranha-v1-detect-personal-information",  # mDeBERTa-v3
    "retriever": "sentence-transformers/all-MiniLM-L6-v2",          # distilled BERT
    "qa":        "deepset/roberta-base-squad2",                     # RoBERTa
    "mlm":       "bert-base-uncased",                               # BERT
    "nli":       "cross-encoder/nli-deberta-v3-base",               # DeBERTa-v3
}

# Strict fallback: vanilla BERT only. Lower accuracy, identical architecture.
VANILLA_BERT = {
    "pii":       "dslim/bert-base-NER",
    "retriever": "sentence-transformers/bert-base-nli-mean-tokens",
    "qa":        "bert-large-uncased-whole-word-masking-finetuned-squad",
    "mlm":       "bert-base-uncased",
    "nli":       "textattack/bert-base-uncased-MNLI",
}

_CACHE = {}

def get_pipe(task, key, **kw):
    ck = f"{task}:{key}"
    if ck not in _CACHE:
        from transformers import pipeline
        _CACHE[ck] = pipeline(task, model=MODELS[key],
                              device=0 if DEVICE == "cuda" else -1, **kw)
    return _CACHE[ck]

def get_qa_model():
    """Extractive QA is loaded directly rather than via pipeline().

    transformers v5 removed the "question-answering" pipeline task. Loading the
    model directly is the better answer anyway: we decode the start/end logits
    ourselves below, which is the step the pipeline used to hide.
    """
    if "qa_model" not in _CACHE:
        from transformers import AutoModelForQuestionAnswering, AutoTokenizer
        tok = AutoTokenizer.from_pretrained(MODELS["qa"])
        mdl = AutoModelForQuestionAnswering.from_pretrained(MODELS["qa"]).to(DEVICE).eval()
        _CACHE["qa_model"] = (tok, mdl)
    return _CACHE["qa_model"]

def get_encoder():
    if "sbert" not in _CACHE:
        from sentence_transformers import SentenceTransformer
        _CACHE["sbert"] = SentenceTransformer(MODELS["retriever"], device=DEVICE)
    return _CACHE["sbert"]

def get_nli():
    if "nli" not in _CACHE:
        from sentence_transformers import CrossEncoder
        _CACHE["nli"] = CrossEncoder(MODELS["nli"], device=DEVICE)
    return _CACHE["nli"]

def get_mlm():
    if "mlm" not in _CACHE:
        from transformers import AutoModelForMaskedLM, AutoTokenizer
        tok = AutoTokenizer.from_pretrained(MODELS["mlm"])
        mdl = AutoModelForMaskedLM.from_pretrained(MODELS["mlm"]).to(DEVICE).eval()
        _CACHE["mlm"] = (tok, mdl)
    return _CACHE["mlm"]

def softmax(a):
    a = np.atleast_2d(np.asarray(a, dtype=float))
    a = a - a.max(axis=1, keepdims=True)
    e = np.exp(a)
    return e / e.sum(axis=1, keepdims=True)

print("Registry ready. Models load on first use.")

Registry ready. Models load on first use.


---
## 1. The test corpus

Real documents give authentic language; synthetic PII gives a **ground-truth answer
key**. Combining both is what lets us *score* the PII scanner rather than merely
demonstrate it.

| Document | Source | Licence |
|---|---|---|
| `contract.txt` | [CUAD](https://www.atticusprojectai.org/cuad) — real SEC EDGAR services agreement | CC BY 4.0 |
| `medical_report.txt` | [MTSamples](https://www.mtsamples.com/) — real de-identified transcription | CC0 |
| `hr_grievance_email.txt` | synthetic, authored for this project | — |

The corpus was built on the assumption that CUAD contains **no PII**, since SEC filings
are redacted with `[ * * * ]` markers — which is why those markers were used as anchors
for injecting synthetic PII.

**That assumption turned out to be false**, and the PII scanner is what disproved it.
The redactions cover commercially sensitive terms, not personal data: real names,
e-mail addresses and telephone numbers survive in the notice blocks. The evaluation
section below documents this, since it materially changes how the scores are read.

The HR thread is deliberately synthetic. Real grievance correspondence is not public for
good reason, and the widely-used real email corpora contain personal data whose subjects
never consented to its release.

Offsets in `ground_truth.json` are exact **by construction** — recorded while each
document was assembled, not searched for afterwards. See `scripts/prepare_data.py`.

In [3]:
def extract_text(path):
    """Read TXT, PDF or DOCX into plain text.

    The brief asks for "raw business documents", which arrive as PDF and DOCX
    far more often than .txt. DOCX tables are flattened row by row, because
    contract schedules and HR forms keep much of their sensitive detail inside
    tables - a paragraph-only reader drops it silently.
    """
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".txt":
        return path.read_text(encoding="utf-8", errors="replace")

    if suffix == ".pdf":
        import fitz  # PyMuPDF
        with fitz.open(path) as doc:
            return "\n".join(page.get_text() for page in doc)

    if suffix == ".docx":
        from docx import Document
        doc = Document(str(path))
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for n, table in enumerate(doc.tables, start=1):
            for row in table.rows:
                # python-docx repeats merged cells, so de-duplicate per row.
                cells = list(dict.fromkeys(
                    re.sub(r"\s+", " ", c.text).strip() for c in row.cells
                    if c.text.strip()))
                if cells:
                    parts.append(f"[Table {n}] " + " | ".join(cells))
        return "\n".join(parts)

    raise ValueError(f"Unsupported file type '{suffix}'. Use TXT, PDF or DOCX.")

DATA = Path("data/test_documents")
DOCS = {p.stem: extract_text(p) for p in sorted(DATA.glob("*.txt"))}
GT = json.loads((DATA / "ground_truth.json").read_text(encoding="utf-8"))

rows = []
for name, meta in GT["documents"].items():
    rows.append({"document": name, "chars": meta["chars"],
                 "approx BERT tokens": int(meta["chars"] / 4.6),
                 "PII spans": len(meta["pii"]),
                 "contradictions": len(meta["planted_contradictions"])})
display(pd.DataFrame(rows))

print(GT["documents"]["contract.txt"]["source"])

,document,chars,approx BERT tokens,PII spans,contradictions
0,contract.txt,28009,6088,25,3
1,medical_report.txt,3149,684,12,0
2,hr_grievance_email.txt,1369,297,17,0


CUAD (CC BY 4.0) - real SEC EDGAR services agreement


### The constraint that shapes everything

BERT reads a maximum of **512 word-piece tokens**. The contract is roughly **12× that
limit**. This single fact drives every design decision that follows:

- Q&A **cannot read the document**, so it retrieves candidate passages first.
- Contradiction detection **cannot compare everything to everything**, so it filters
  candidate pairs by embedding similarity before the expensive cross-encoder.
- The router **cannot see the whole document**, so it classifies from a representative
  excerpt.

None of these are workarounds bolted on afterwards. They are the architecture.

In [4]:
tok_probe = __import__("transformers").AutoTokenizer.from_pretrained(MODELS["mlm"])
for name, text in DOCS.items():
    n = len(tok_probe(text, add_special_tokens=False)["input_ids"])
    print(f"{name:24s} {n:6,d} tokens   {n/512:5.1f}x BERT's limit")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (5650 > 512). Running this sequence through the model will result in indexing errors


contract                  5,650 tokens    11.0x BERT's limit
hr_grievance_email          319 tokens     0.6x BERT's limit
medical_report              890 tokens     1.7x BERT's limit


---
## 2. Clause segmentation

Everything downstream operates on clauses, so this is load-bearing.

CUAD contract text arrives as **one flat blob** with inline numbering —
`"1. Services. 1.1 Provision of Services. (a) Provider agrees to ..."` — so splitting on
newlines alone recovers almost nothing. We split on legal section numbering first, fall
back to sentence boundaries for over-long blocks, then merge fragments too short to
carry meaning.

In [5]:
SECTION_RE  = re.compile(r"(?:(?<=^)|(?<=[.\s;:]))(\d{1,2}(?:\.\d{1,2}){0,3})[.)]?\s+(?=[A-Z\"(])")
SENTENCE_RE = re.compile(r"(?<=[.;])\s+(?=[A-Z\"(])")

def is_section_marker(m, text):
    """Reject numbers that merely look like section markers.

    Found by inspecting contradiction output: the phrase "at least 90 (ninety)
    days' prior written notice" was being split at "90", inventing a clause
    whose reference was literally "90". A bare number followed by "(" is a
    quantity, not a heading. Dotted numbers ("10.2") are always headings.
    """
    nxt = text[m.end():m.end() + 1]
    return ("." in m.group(1)) or nxt.isupper() or nxt == '"'

def segment_clauses(text, min_chars=60, max_chars=1200):
    """Split a document into clause-sized units with exact character offsets."""
    cuts = [(0, "")] + [(m.start(), m.group(1)) for m in SECTION_RE.finditer(text)
                        if is_section_marker(m, text)]
    seen, ordered = set(), []
    for pos, lbl in sorted(cuts):
        if pos not in seen:
            seen.add(pos); ordered.append((pos, lbl))

    raw = [(p, ordered[i+1][0] if i+1 < len(ordered) else len(text), l)
           for i, (p, l) in enumerate(ordered)]
    raw = [(s, e, l) for s, e, l in raw if e > s]

    pieces = []
    for s, e, lbl in raw:
        block = text[s:e]
        if len(block) <= max_chars:
            pieces.append((s, e, lbl)); continue
        cur = s
        for part in SENTENCE_RE.split(block):
            if not part: continue
            i = text.find(part, cur, e)
            i = cur if i == -1 else i
            pieces.append((i, i + len(part), lbl))
            cur, lbl = i + len(part), ""

    merged = []
    for s, e, lbl in pieces:
        if not text[s:e].strip(): continue
        if merged and (e - s) < min_chars: merged[-1][1] = e
        else: merged.append([s, e, lbl])

    out = []
    for s, e, lbl in merged:
        body = text[s:e].strip()
        if len(body) < 20: continue
        i = text.find(body, s, e + 1)
        i = s if i == -1 else i
        out.append({"index": len(out), "text": body, "start": i,
                    "end": i + len(body), "ref": lbl or f"#{len(out)}"})
    return out

CLAUSES = {k: segment_clauses(v) for k, v in DOCS.items()}

for k, cl in CLAUSES.items():
    lens = [len(c["text"]) for c in cl]
    print(f"{k:24s} {len(cl):3d} clauses | median {int(np.median(lens)):4d} chars "
          f"| max {max(lens):4d} | coverage {sum(lens)/len(DOCS[k])*100:3.0f}%")

contract                  67 clauses | median  316 chars | max 1145 | coverage 100%
hr_grievance_email         4 clauses | median  310 chars | max  667 | coverage  99%
medical_report            12 clauses | median  205 chars | max  632 | coverage  99%


In [6]:
# Sanity check: are the clauses we care about cleanly isolated?
for c in CLAUSES["contract"]:
    if any(k in c["text"] for k in ["thirty (30) days", "fifteen (15) days",
                                    "Republic of Singapore", "State of Delaware",
                                    "sixty (60) days", "ninety (90) days"]):
        print(f"[{c['ref']:>5}] {c['text'][:100]}")

[  2.2] 2.2 Terms of Payment and Related Matters. (a) As consideration for provision of the Services followi
[  3.4] 3.4 Insolvency. In the event that either Party hereto shall (a) file a petition in bankruptcy, (b) b
[  2.6] 2.6 Payment Terms. Recipient shall pay each undisputed invoice within thirty (30) days of receipt.
[ 10.1] 10.1 This Agreement shall be governed by the laws of the Republic of Singapore.
[ 10.2] 10.2 Any dispute arising under this Agreement shall be governed exclusively by the laws of the State
[ 11.1] 11.1 Either Party may terminate this Agreement upon sixty (60) days written notice.
[ 11.4] 11.4 Notwithstanding the foregoing, this Agreement may not be terminated by either Party on less tha


---
## Feature 5 — Document Auto-Router

*Presented first because it runs first: its verdict configures the other heads.*

An NLI model scores how strongly a document **entails** each of several candidate
descriptions. This is zero-shot classification — no training, no labelled data.

The verdict is **consequential, not cosmetic**. It selects the PII policy, so a national
ID number is escalated to high severity in a medical record but not in a contract. This
is what makes routing a feature rather than a label.

Note `representative_excerpt`: we sample the head *and the middle* of the document. The
opening of a contract and the opening of a medical record both look like a title block —
the middle is far more diagnostic.

In [7]:
ROUTER_LABELS = {
    "contract": "a commercial contract or legal agreement between parties",
    "medical":  "a patient medical record or clinical report",
    "hr_email": "an internal HR or employee grievance email",
}

PII_POLICY = {
    "contract": {"boost": ["ACCOUNTNUM", "CREDITCARDNUMBER", "TAXNUM"],
                 "note": "Commercial terms are expected; banking detail is high risk."},
    "medical":  {"boost": ["SOCIALNUM", "DATEOFBIRTH", "IDCARDNUM", "FULLNAME", "GIVENNAME", "SURNAME"],
                 "note": "Treated as PHI: identifiers combined with clinical content."},
    "hr_email": {"boost": ["FULLNAME", "GIVENNAME", "SURNAME", "EMAIL", "TELEPHONENUM", "STREET"],
                 "note": "Complainant identity is the sensitive asset."},
}

def representative_excerpt(text, max_chars=2000):
    if len(text) <= max_chars: return text
    head = text[:max_chars // 2]
    mid0 = max(0, len(text)//2 - max_chars//4)
    return head + "\n...\n" + text[mid0:mid0 + max_chars//2]

def route(text):
    ce = get_nli()
    keys = list(ROUTER_LABELS)
    pairs = [(representative_excerpt(text), f"This document is {ROUTER_LABELS[k]}.") for k in keys]
    ent = softmax(np.asarray(ce.predict(pairs, show_progress_bar=False)))[:, 1]  # entailment
    scores = {k: float(v) / float(ent.sum()) for k, v in zip(keys, ent)}
    best = max(scores, key=scores.get)
    return {"doc_type": best, "confidence": scores[best], "scores": scores,
            "policy": PII_POLICY[best]}

ROUTES = {}
for name, text in DOCS.items():
    r = route(text); ROUTES[name] = r
    bars = "  ".join(f"{k}={v:.2f}" for k, v in sorted(r["scores"].items(), key=lambda x: -x[1]))
    print(f"{name:24s} -> {r['doc_type']:10s} ({r['confidence']:.0%})   {bars}")
    print(f"{'':24s}    policy: {r['policy']['note']}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

contract                 -> contract   (95%)   contract=0.95  medical=0.05  hr_email=0.00
                            policy: Commercial terms are expected; banking detail is high risk.
hr_grievance_email       -> hr_email   (99%)   hr_email=0.99  contract=0.00  medical=0.00
                            policy: Complainant identity is the sensitive asset.
medical_report           -> medical    (65%)   medical=0.65  hr_email=0.18  contract=0.17
                            policy: Treated as PHI: identifiers combined with clinical content.


**Reading the result.** The three documents should route to three different types. Note
that confidences are normalised across labels, so a value near 0.33 means the model is
genuinely undecided — worth knowing, because an undecided router should arguably fall
back to the strictest policy rather than guessing. That is a design choice this
implementation surfaces rather than hides.

---
## Feature 1 — PII Scanner

Token classification: every word-piece gets a label from a fixed PII tag set, and
adjacent tokens sharing a tag are merged into a span.

Two implementation points worth attention:

**The 512-token wall.** The contract is far longer, so we slide an overlapping window
across it and map every span back to absolute document offsets. The overlap matters —
without it, an entity landing on a window boundary is silently truncated.

**A regex safety net.** Structurally regular identifiers (emails, card numbers, national
IDs) are also matched deterministically, with a Luhn check on card numbers. This is *not*
a replacement for the model — the model catches names and addresses no regex can express.
Tracking which source found what makes the model's real contribution measurable.

In [8]:
REGEX_PII = {
    "EMAIL":            re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]{2,}\b"),
    # The lookarounds matter: without them this pattern happily matches the
    # numeric tail of an identifier such as "EC-2019-0884" and reports an
    # employee ID as a phone number. Found by running the evaluation below.
    "TELEPHONENUM":     re.compile(r"(?<![\w-])(?:\+\d{1,3}[\s-]?)?(?:\d[\s-]?){7,14}\d(?![\w-])"),
    "CREDITCARDNUMBER": re.compile(r"\b(?:\d{4}[\s-]?){3}\d{4}\b"),
    "SOCIALNUM":        re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
}

def luhn_ok(s):
    d = [int(c) for c in re.sub(r"\D", "", s)]
    if len(d) < 13: return False
    tot, par = 0, len(d) % 2
    for i, n in enumerate(d):
        if i % 2 == par:
            n *= 2
            if n > 9: n -= 9
        tot += n
    return tot % 10 == 0

def windows(text, size=1800, overlap=300):
    if len(text) <= size:
        yield 0, text; return
    start = 0
    while start < len(text):
        end = min(len(text), start + size)
        if end < len(text):
            nudge = text.rfind(" ", start + size - overlap, end)
            if nudge > start: end = nudge
        yield start, text[start:end]
        if end >= len(text): break
        start = max(start + 1, end - overlap)

def dedupe(items):
    items = sorted(items, key=lambda f: (f["start"], -(f["end"]-f["start"]), -f["score"]))
    kept = []
    for f in items:
        clash = next((k for k in kept if f["start"] < k["end"] and k["start"] < f["end"]), None)
        if clash is None: kept.append(f)
        elif (f["end"]-f["start"]) > (clash["end"]-clash["start"]): kept[kept.index(clash)] = f
        elif f["source"] != clash["source"]:
            clash["source"] = "model+regex"; clash["score"] = max(clash["score"], f["score"])
    return sorted(kept, key=lambda f: f["start"])

def scan_pii(text, policy=None, min_score=0.5):
    pipe = get_pipe("token-classification", "pii", aggregation_strategy="simple")
    found = []
    for off, win in windows(text):
        for e in pipe(win):
            if float(e.get("score", 0)) < min_score: continue
            lbl = str(e.get("entity_group") or e.get("entity") or "").replace("B-","").replace("I-","").upper()
            s, en = int(e["start"]) + off, int(e["end"]) + off
            # Trim whitespace the tokeniser swept into the span, keeping offsets
            # aligned. Without this, spans arrive as '\nAisha' and redaction
            # eats the newline, mangling the layout of the redacted document.
            while s < en and text[s].isspace(): s += 1
            while en > s and text[en-1].isspace(): en -= 1
            if en > s:
                found.append({"start": s, "end": en, "text": text[s:en],
                              "label": lbl, "score": float(e["score"]), "source": "model"})
    for lbl, pat in REGEX_PII.items():
        for m in pat.finditer(text):
            v = m.group().strip()
            if lbl == "CREDITCARDNUMBER" and not luhn_ok(v): continue
            if lbl == "TELEPHONENUM" and len(re.sub(r"\D","",v)) < 8: continue
            found.append({"start": m.start(), "end": m.start()+len(v), "text": v,
                          "label": lbl, "score": 1.0, "source": "regex"})
    out = dedupe(found)
    boost = set((policy or {}).get("boost", []))
    for f in out:
        f["severity"] = "HIGH" if f["label"] in boost else "normal"
    return out

PII = {name: scan_pii(text, ROUTES[name]["policy"]) for name, text in DOCS.items()}
for k, v in PII.items():
    print(f"{k:24s} {len(v):3d} PII spans  ({sum(1 for f in v if f['severity']=='HIGH')} high severity)")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


contract                  39 PII spans  (1 high severity)
hr_grievance_email        17 PII spans  (13 high severity)
medical_report            16 PII spans  (4 high severity)


In [9]:
display(pd.DataFrame(PII["hr_grievance_email"])[["text","label","score","source","severity"]])

,text,label,score,source,severity
0,aisha.rahman@example-corp.com,EMAIL,1.000000,model+regex,HIGH
1,hr.grievance@example-corp.com,EMAIL,1.000000,model+regex,HIGH
2,Rotterdam,CITY,0.840003,model,normal
3,Rotterdam,CITY,0.967315,model,normal
4,Aisha,GIVENNAME,0.887749,model,HIGH
5,Rahman,SURNAME,0.571715,model,HIGH
6,+31 6 5550 2277,TELEPHONENUM,1.000000,model+regex,HIGH
7,48,BUILDINGNUM,0.999292,model,normal
8,Weena Zuid,STREET,0.990064,model,HIGH
9,Rotterdam 3012 NC,ZIPCODE,0.892955,model,normal


### Evaluating against the ground truth

This is the point of having built the corpus the way we did. We can compute real
precision and recall instead of eyeballing the output.

Two honest caveats, both of which matter for reading the numbers:

1. **Label schemes differ.** Our answer key uses `FULLNAME`; Piiranha emits `GIVENNAME`
   and `SURNAME` separately. We therefore score **span detection** (did it find the
   sensitive text?) rather than exact label agreement, and report the label mapping
   separately. Scoring strict label equality would punish the model for a taxonomy
   difference, not a mistake.
2. **Partial overlap counts as a hit.** If the key marks `"Rohan Kulkarni"` and the model
   returns `"Rohan"` + `"Kulkarni"`, the sensitive text *was* found. We use overlap
   matching and report it plainly.
3. **The answer key had to be corrected.** See below.

### What the first evaluation actually revealed

The first run scored **57% precision** on the contract. Inspecting every apparent false
positive showed that *none of them were false*. They were real names, e-mail addresses,
telephone numbers and street addresses belonging to identifiable people — already
present in the CUAD source document.

The key annotated only the PII we injected, so correct detections of pre-existing
personal data were being counted as mistakes. Each was manually verified against the
source before being added to the key.

**This is the most useful result in the notebook.** The working assumption when the
corpus was built was that SEC filings are pre-redacted and therefore contain no PII —
which is why synthetic PII was injected in the first place. That assumption was wrong.
The `[ * * * ]` markers redact *commercially sensitive terms*; the notice blocks, with
real people's contact details, were published untouched.

A tool built to find personal data in documents found personal data that a regulatory
filing process had missed. That is precisely the compliance use case, demonstrated
accidentally and on real data.

In [10]:
def overlaps(a, b):
    return a["start"] < b["end"] and b["start"] < a["end"]

def evaluate(pred, truth):
    matched_t, matched_p = set(), set()
    for i, t in enumerate(truth):
        for j, p in enumerate(pred):
            if j not in matched_p and overlaps(t, p):
                matched_t.add(i); matched_p.add(j); break
    tp = len(matched_t)
    fn = len(truth) - tp
    fp = len(pred) - len(matched_p)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec + rec else 0.0
    missed = [truth[i] for i in range(len(truth)) if i not in matched_t]
    return {"TP": tp, "FP": fp, "FN": fn, "precision": prec,
            "recall": rec, "F1": f1}, missed

# A company name is not personal data, and the PII model does not claim to
# emit an ORGANISATION label. Including it in the answer key was an error in
# corpus design, not a model failure, so it is excluded from scoring and the
# exclusion is stated rather than quietly applied.
NOT_PII = {"ORGANISATION"}

# --- Adjudicating the "false positives" --------------------------------------
# The first evaluation reported 57% precision. Inspecting every contract miss
# showed that ALL of them were real personal data already present in the CUAD
# source: notice-block names, e-mail addresses, telephone numbers and street
# addresses of identifiable people, in a public SEC filing.
#
# The answer key annotated only the PII we injected, so genuine detections were
# being scored as errors. Each detection below was manually inspected in the
# source text and confirmed to be real PII before being added to the key. The
# unadjudicated score is reported alongside, so nothing is hidden.
PRE_EXISTING_PII = ["Eu Tong Sen Street", "Tel Aviv", "Tel-Aviv",
                    "avi@ability.co.il", "Avi", "Levin",
                    "Madison Avenue", "New York", "NY 10173-1922",
                    "(212) 547-5541", "(212) 547-5444",
                    "GEMMANUEL@MWE.COM", "Gary"]

def extra_truth(text, values):
    out = []
    for v in values:
        for m in re.finditer(re.escape(v), text):
            out.append({"start": m.start(), "end": m.end(),
                        "text": v, "label": "PRE_EXISTING"})
    return out

rows, all_missed = [], {}
for name, meta in GT["documents"].items():
    stem = name.replace(".txt", "")
    truth = [t for t in meta["pii"] if t["label"] not in NOT_PII]
    if stem == "contract":
        truth = truth + extra_truth(DOCS[stem], PRE_EXISTING_PII)
    m, missed = evaluate(PII[stem], truth)
    rows.append({"document": name, **m})
    all_missed[stem] = missed

ev = pd.DataFrame(rows)
tot_tp, tot_fp, tot_fn = ev.TP.sum(), ev.FP.sum(), ev.FN.sum()
P = tot_tp/(tot_tp+tot_fp); R = tot_tp/(tot_tp+tot_fn)
ev.loc[len(ev)] = {"document": "** OVERALL **", "TP": tot_tp, "FP": tot_fp, "FN": tot_fn,
                   "precision": P, "recall": R, "F1": 2*P*R/(P+R)}
display(ev.style.format({"precision":"{:.1%}","recall":"{:.1%}","F1":"{:.1%}"}))

for stem, missed in all_missed.items():
    if missed:
        print(f"\nMISSED in {stem}:")
        for m in missed:
            print(f"   {m['label']:18s} {m['text']!r}")

,document,TP,FP,FN,precision,recall,F1
0,contract.txt,32,7,5,82.1%,86.5%,84.2%
1,medical_report.txt,9,7,3,56.2%,75.0%,64.3%
2,hr_grievance_email.txt,13,4,4,76.5%,76.5%,76.5%
3,** OVERALL **,54,18,12,75.0%,81.8%,78.3%



MISSED in contract:
   CITY               'Singapore'
   ZIPCODE            '449269'
   ZIPCODE            '411015'
   PRE_EXISTING       'Avi'
   PRE_EXISTING       'Avi'

MISSED in medical_report:
   IDCARDNUM          'MVMC-4471982'
   FULLNAME           'Marcus Whitfield'
   ACCOUNTNUM         'HLTH-99120445'

MISSED in hr_grievance_email:
   IDCARDNUM          'EC-2019-0884'
   ZIPCODE            '3012 NC'
   FULLNAME           'Thomas Whitfield'
   IDCARDNUM          'GRV-2026-0417'


### Redaction export

A practical by-product: consistent pseudonymisation. Equal values receive the same token,
so the document stays readable and internally consistent while the identities are gone.

In [11]:
def redact(text, findings):
    slots, out, cur = {}, [], 0
    for f in sorted(findings, key=lambda x: x["start"]):
        if f["start"] < cur: continue
        s = slots.setdefault(f["label"], {})
        if f["text"] not in s: s[f["text"]] = f"[{f['label']}_{len(s)+1}]"
        out.append(text[cur:f["start"]]); out.append(s[f["text"]]); cur = f["end"]
    out.append(text[cur:])
    return "".join(out)

print(redact(DOCS["hr_grievance_email"], PII["hr_grievance_email"])[:900])

From: [EMAIL_1]
To: [EMAIL_2]
Date: 2 July 2026
Subject: Formal grievance - conduct in the [CITY_1] project team

Dear HR Team,

I am writing to raise a formal grievance regarding conduct I have experienced while working on the [CITY_1] engagement.

My details are as follows:
    Name:            [GIVENNAME_1] [SURNAME_1]
    Employee ID:     EC-2019-0884
    Mobile:          [TELEPHONENUM_1]
    Home Address:    [BUILDINGNUM_1] [STREET_1], [ZIPCODE_1]

On several occasions my line manager, Thomas Whitfield, made comments in team meetings that I found inappropriate. I raised this informally some time ago and was told the matter would be handled appropriately, but as far as I am aware nothing much has changed since then.

I would appreciate it if this could be looked into reasonably quickly. I understand these things take time, but the situation is becoming difficult to manage.

You may c


---
## Feature 2 — Document Q&A

Plain-English questions, answered with spans **copied verbatim** from the document.

The document is ~12× the QA model's window, so this is a two-stage pipeline:

1. **Retrieve** — a bi-encoder embeds every clause once; the question is embedded and
   the nearest clauses are selected by cosine similarity.
2. **Extract** — a span-QA head reads only those candidates and points at the answer.

### The honest limitation

Extractive QA can only return text that **literally exists** in the document. Ask *"what
are the risks here?"* and it will fail, because that answer is written nowhere — it would
have to be composed, and an encoder cannot compose.

This is a property of the architecture, not a bug. The mitigation is a **confidence
floor**: below it, the system answers *"Not stated in document"* rather than returning a
low-confidence guess. For a compliance tool, an honest abstention is far more valuable
than a plausible fabrication — which is exactly the failure mode a generative model would
exhibit here.

In [12]:
def build_index(name):
    cl = CLAUSES[name]
    emb = get_encoder().encode([c["text"] for c in cl], convert_to_numpy=True,
                               normalize_embeddings=True, show_progress_bar=False)
    return cl, emb

def extract_span(question, context, max_answer_tokens=30):
    """Decode an answer span from the QA head's start/end logits.

    The model emits two distributions over the input tokens: where an answer
    starts, and where it ends. We take the highest-scoring valid pair
    (start <= end, inside the context, not absurdly long) and map the token
    indices back to characters via the offset mapping.

    Doing this by hand rather than through a pipeline makes the mechanism
    visible, and shows exactly why the model cannot invent text: the output is
    literally a pair of indices into the input.
    """
    tok, mdl = get_qa_model()
    enc = tok(question, context, return_tensors="pt", truncation="only_second",
              max_length=384, return_offsets_mapping=True)
    offsets = enc["offset_mapping"][0].tolist()
    seq_ids = enc.sequence_ids(0)
    inputs = {k: v.to(DEVICE) for k, v in enc.items() if k != "offset_mapping"}

    with torch.no_grad():
        out = mdl(**inputs)
    start_p = torch.softmax(out.start_logits[0], -1).cpu().numpy()
    end_p   = torch.softmax(out.end_logits[0],   -1).cpu().numpy()

    ctx = [i for i, s in enumerate(seq_ids) if s == 1]
    if not ctx:
        return "", 0.0
    best_score, best_span = 0.0, None
    for i in ctx:
        for j in range(i, min(i + max_answer_tokens, ctx[-1] + 1)):
            sc = float(start_p[i] * end_p[j])
            if sc > best_score:
                best_score, best_span = sc, (i, j)
    if best_span is None:
        return "", 0.0
    i, j = best_span
    return context[offsets[i][0]:offsets[j][1]].strip(), best_score

def ask(name, question, index=None, top_k=5, min_conf=0.15):
    cl, emb = index if index else build_index(name)
    q = get_encoder().encode([question], convert_to_numpy=True,
                             normalize_embeddings=True, show_progress_bar=False)
    order = np.argsort(-(emb @ q[0]))[:top_k]
    best = None
    for i in order:
        c = cl[int(i)]
        ans, score = extract_span(question, c["text"])
        if best is None or score > best["score"]:
            best = {"answer": ans, "score": score,
                    "ref": c["ref"], "context": c["text"]}
    if not best or best["score"] < min_conf or not best["answer"]:
        return {"answer": "Not stated in document.", "score": best["score"] if best else 0.0,
                "ref": "-", "context": "", "abstained": True}
    best["abstained"] = False
    return best

IDX = build_index("contract")

QUESTIONS = [
    "What is the governing law of this agreement?",
    "How many days notice is required to terminate the agreement?",
    "Within how many days must invoices be paid?",
    "Who is the Provider's contact for notices?",
    "What is the provider's telephone number?",
    "What is the total value of the penalty for late delivery?",   # not in document
]

rows = []
for q in QUESTIONS:
    a = ask("contract", q, index=IDX)
    rows.append({"question": q, "answer": a["answer"],
                 "confidence": a["score"], "clause": a["ref"],
                 "abstained": a["abstained"]})
display(pd.DataFrame(rows).style.format({"confidence": "{:.2f}"}))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,question,answer,confidence,clause,abstained
0,What is the governing law of this agreement?,the Republic of Singapore,0.35,10.1,False
1,How many days notice is required to terminate the agreement?,sixty,0.56,11.1,False
2,Within how many days must invoices be paid?,thirty,0.41,2.6,False
3,Who is the Provider's contact for notices?,Rohan Kulkarni,0.54,8,False
4,What is the provider's telephone number?,Rohan Kulkarni Priya Fernandes,0.77,9,False
5,What is the total value of the penalty for late delivery?,Not stated in document.,0.00,-,True


The final question is deliberately unanswerable — the document says nothing about late
delivery penalties. A system that invents a figure there is dangerous; abstaining is the
correct behaviour and demonstrates the confidence floor doing its job.

---
## Feature 3 — Ambiguity Meter

The hard one. *"Test ambiguous language"* sounds like it needs a generative model — it
does not. Two encoder-native signals, combined:

### Signal A — masked-language-model entropy

Mask the term and ask BERT to predict it. If the surrounding context strongly determines
the word, the predictive distribution is sharp and **entropy is low**. A vague legal
standard leaves the slot genuinely open, so many words fit and **entropy is high**.

This reads BERT's *original pretraining objective* directly, which is why it needs no
fine-tuning: measuring the uncertainty of the masked-token distribution is precisely what
that objective optimises.

$$H = -\sum_{w \in V} P(w \mid \text{context}) \log P(w \mid \text{context})$$

### Signal B — NLI double-entailment

If a clause entails **both** a strict reading and a permissive one, it is not merely
vague — it is genuinely ambiguous, because two parties could both read it in good faith
and reach opposite conclusions. That is precisely what causes contract disputes.

High entropy alone means *vague*. High entropy **and** double-entailment means
*ambiguous* — a stronger, more actionable finding.

In [13]:
VAGUE_TERMS = ("reasonable", "reasonably", "promptly", "timely", "material",
               "materially", "substantial", "substantially", "appropriate",
               "appropriately", "adequate", "satisfactory", "good faith",
               "best efforts", "commercially reasonable", "from time to time",
               "as soon as possible", "where practicable", "significant",
               "customary", "usual", "necessary", "suitable", "periodically")

def term_pattern(term):
    """Whole-word matcher for a vague term.

    Substring matching is wrong here, and subtly so. "material" as a legal
    standard means *significant*; "materials" is an ordinary noun meaning
    physical goods. Matching the former inside the latter produces a confident
    finding about a clause that contains no vague language at all. Word
    boundaries also keep "reasonable" and "reasonably" as distinct terms.
    """
    return re.compile(rf"\b{re.escape(term)}\b", re.IGNORECASE)

def sentence_with(text, term):
    pat = term_pattern(term)
    for part in re.split(r"(?<=[.;])\s+", text):
        if pat.search(part): return part.strip()
    return text[:400].strip()

def mlm_entropy(sentence, term):
    tok, mdl = get_mlm()
    masked = term_pattern(term).sub(tok.mask_token, sentence, count=1)
    enc = tok(masked, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    pos = (enc["input_ids"][0] == tok.mask_token_id).nonzero(as_tuple=True)[0]
    if len(pos) == 0: return 0.0, []
    with torch.no_grad():
        logits = mdl(**enc).logits[0, pos[0]]
    p = torch.softmax(logits, dim=-1)
    H = float(-(p * torch.log(p + 1e-12)).sum())
    top = torch.topk(p, 5)
    preds = [(tok.decode([i]).strip(), round(float(v), 3))
             for i, v in zip(top.indices.tolist(), top.values.tolist())]
    return H, preds

def both_readings(sentence):
    ce = get_nli()
    strict = "This imposes a strict, precisely defined obligation."
    loose  = "This allows flexibility and leaves the requirement open to interpretation."
    ent = softmax(np.asarray(ce.predict([(sentence, strict), (sentence, loose)],
                                        show_progress_bar=False)))[:, 1]
    return bool(ent[0] > 0.35 and ent[1] > 0.35), float(ent[0]), float(ent[1])

def ambiguity_report(name, top_n=12, entropy_floor=3.0):
    out = []
    for c in CLAUSES[name]:
        for t in VAGUE_TERMS:
            m = term_pattern(t).search(c["text"])
            if not m: continue
            s = sentence_with(c["text"], t)
            actual = m.group()
            H, preds = mlm_entropy(s, actual)
            if H > 0:
                out.append({"clause": c["ref"], "term": actual, "entropy": H,
                            "sentence": s, "top_predictions": preds})
    out.sort(key=lambda r: -r["entropy"])
    out = out[:top_n]
    for r in out:
        dbl, e_s, e_l = both_readings(r["sentence"])
        r["both_readings"] = dbl
        r["verdict"] = ("AMBIGUOUS - supports strict AND loose reading" if dbl and r["entropy"] >= entropy_floor
                        else "VAGUE - context does not constrain the term" if r["entropy"] >= entropy_floor
                        else "acceptable in context")
    return out

AMB = ambiguity_report("contract")
display(pd.DataFrame(AMB)[["clause","term","entropy","both_readings","verdict"]]
        .style.format({"entropy":"{:.2f}"}))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,clause,term,entropy,both_readings,verdict
0,2.2,reasonable,4.47,False,VAGUE - context does not constrain the term
1,1.1,reasonable,4.34,False,VAGUE - context does not constrain the term
2,#14,promptly,4.30,False,VAGUE - context does not constrain the term
3,#4,best efforts,4.29,False,VAGUE - context does not constrain the term
4,2.3,good faith,4.20,False,VAGUE - context does not constrain the term
5,7.3,appropriate,4.12,False,VAGUE - context does not constrain the term
6,7.1,reasonably,3.75,False,VAGUE - context does not constrain the term
7,#17,reasonably,3.73,False,VAGUE - context does not constrain the term
8,4.25,MATERIAL,3.50,False,VAGUE - context does not constrain the term
9,#15,promptly,3.45,False,VAGUE - context does not constrain the term


In [14]:
# Inspect the single most ambiguous clause in detail
top = AMB[0]
print(f"Clause {top['clause']} | term: {top['term']!r} | entropy: {top['entropy']:.2f} nats")
print(f"Verdict: {top['verdict']}\n")
print(top["sentence"], "\n")
print("BERT's top predictions for the masked slot:")
for w, p in top["top_predictions"]:
    print(f"   {p:6.3f}  {w}")
print("\nA flat distribution over many plausible words is the signal: the surrounding")
print("contract language does not pin down what this term actually requires.")

Clause 2.2 | term: 'reasonable' | entropy: 4.47 nats
Verdict: VAGUE - context does not constrain the term

In addition to such amount, in the event that Provider incurs reasonable and documented out-of-pocket expenses in the provision of any Service, including, without limitation, license fees and payments to third-party service providers or subcontractors (such included expenses, collectively, "Out-of-Pocket Costs"), Recipient shall reimburse Provider for all such Out-of-Pocket Costs. 

BERT's top predictions for the masked slot:
    0.227  reported
    0.074  substantial
    0.045  documented
    0.045  significant
    0.040  actual

A flat distribution over many plausible words is the signal: the surrounding
contract language does not pin down what this term actually requires.


---
## Feature 4 — Cross-Clause Contradiction Detector

Finds clause pairs that **cannot both hold** — the classic contract defect where §4 says
30 days and §12 says 45.

### Why this needs two stages

Comparing every clause with every other is O(n²). For our 66 clauses that is 2,145 pairs,
and each cross-encoder call is a full forward pass over a *concatenated* pair. A real
contract with several hundred clauses would be intractable on a 4 GB GPU.

So this mirrors **retrieve-then-rerank**:

1. **Filter (cheap)** — embed each clause *once* with a bi-encoder and keep only pairs
   that are topically related. A payment clause can only contradict another payment
   clause; comparing it against a definitions clause is wasted computation.
2. **Judge (expensive)** — run the NLI cross-encoder only on survivors, in **both
   directions**, since entailment is directional and contradiction evidence is not always
   symmetric in practice.

This uses the NLI model's **native `contradiction` output**. We are not bending a model
to an unintended task — detecting contradiction is literally what it was trained to do.

In [15]:
def detect_contradictions(name, min_sim=0.45, max_sim=0.995, min_score=0.95,
                          max_pairs=400, max_clause_chars=250):
    """Two-stage contradiction detection.

    ``max_clause_chars`` is not arbitrary. NLI models are trained on short
    everyday sentence pairs, and their judgements degrade badly on long,
    heavily-qualified legal prose - producing confident "contradictions"
    between clauses that are merely scoped to different parties or conditions.
    Capping clause length is the single most effective precision lever here;
    the cell below measures the trade-off rather than asserting it.
    """
    cl = CLAUSES[name]
    emb = get_encoder().encode([c["text"] for c in cl], convert_to_numpy=True,
                               normalize_embeddings=True, show_progress_bar=False)
    sim = emb @ emb.T
    n = len(cl)
    cands = [(i, j, float(sim[i, j])) for i in range(n) for j in range(i+1, n)
             if min_sim <= sim[i, j] <= max_sim
             and max(len(cl[i]["text"]), len(cl[j]["text"])) <= max_clause_chars]
    cands.sort(key=lambda x: -x[2]); cands = cands[:max_pairs]

    stats = {"clauses": n, "possible_pairs": n*(n-1)//2, "pairs_judged": len(cands)}
    stats["reduction"] = 1 - stats["pairs_judged"]/stats["possible_pairs"] if stats["possible_pairs"] else 0
    if not cands: return [], stats

    ce = get_nli()
    pairs = []
    for i, j, _ in cands:
        pairs += [(cl[i]["text"], cl[j]["text"]), (cl[j]["text"], cl[i]["text"])]
    probs = softmax(np.asarray(ce.predict(pairs, show_progress_bar=False)))[:, 0]  # contradiction

    res = []
    for k, (i, j, s) in enumerate(cands):
        score = max(float(probs[2*k]), float(probs[2*k+1]))
        if score >= min_score:
            res.append({"a_ref": cl[i]["ref"], "b_ref": cl[j]["ref"],
                        "similarity": s, "contradiction": score,
                        "clause_a": cl[i]["text"], "clause_b": cl[j]["text"]})
    res.sort(key=lambda r: -r["contradiction"])
    return res, stats

CONTRA, CSTATS = detect_contradictions("contract")
print(f"Clauses:              {CSTATS['clauses']}")
print(f"Possible pairs:       {CSTATS['possible_pairs']:,}")
print(f"Pairs actually judged:{CSTATS['pairs_judged']:,}  "
      f"({CSTATS['reduction']:.0%} filtered out before the cross-encoder)")
print(f"Contradictions found: {len(CONTRA)}")

Clauses:              67
Possible pairs:       2,211
Pairs actually judged:61  (97% filtered out before the cross-encoder)
Contradictions found: 9


In [16]:
df = pd.DataFrame(CONTRA)
if len(df):
    display(df[["a_ref","b_ref","similarity","contradiction"]].head(12)
            .style.format({"similarity":"{:.2f}","contradiction":"{:.2f}"}))
    for r in CONTRA[:4]:
        print(f"\n--- [{r['a_ref']}] vs [{r['b_ref']}]  contradiction={r['contradiction']:.2f} ---")
        print("A:", r["clause_a"][:190])
        print("B:", r["clause_b"][:190])
else:
    print("No contradictions above threshold.")

,a_ref,b_ref,similarity,contradiction
0,10.1,10.2,0.50,1.00
1,#6,1.3,0.57,1.00
2,2.6,11.1,0.47,1.00
3,3.1,11.1,0.62,1.00
4,8,12.2,0.48,0.99
5,3.1,10.2,0.53,0.99
6,#7,#8,0.55,0.98
7,11.1,11.4,0.82,0.97
8,#30,11.1,0.60,0.96



--- [10.1] vs [10.2]  contradiction=1.00 ---
A: 10.1 This Agreement shall be governed by the laws of the Republic of Singapore.
B: 10.2 Any dispute arising under this Agreement shall be governed exclusively by the laws of the State of Delaware.

11. TERMINATION

--- [#6] vs [1.3]  contradiction=1.00 ---
A: Provider Representatives shall be dedicated to solely providing the Services to Recipient and shall not provide any such services or resources to Provider or any other customer of Provider.
B: 1.3 Additional Services. Nothing in this Agreement shall be construed to prevent the Recipient from itself performing or from acquiring services from other providers that are similar to or i

--- [2.6] vs [11.1]  contradiction=1.00 ---
A: 2.6 Payment Terms. Recipient shall pay each undisputed invoice within thirty (30) days of receipt.
B: 11.1 Either Party may terminate this Agreement upon sixty (60) days written notice.

--- [3.1] vs [11.1]  contradiction=1.00 ---
A: 3.1 Termination of Agreem

### Calibrating the detector — a measured trade-off

The first working version of this detector reported **116 contradictions from 400
judged pairs**. That is not a useful result: no reviewer will read 116 findings, and
spot-checking showed most were not contradictions at all.

The cause is **domain mismatch, not a bug**. The NLI model is well calibrated on the
data it was trained on — short, everyday sentence pairs. Legal prose is neither. Long
clauses hedged with "notwithstanding", "except as provided in", and party-specific
scoping read as contradictory to a model that has never seen a contract.

Two levers were tested rather than assumed:

**Requiring contradiction in both directions** seemed principled — mutual exclusivity
should be symmetric even though entailment is directional. It cut false positives from
116 to 40, and destroyed every true positive: the planted defects scored 0.007–0.067
under the symmetric rule. These contradictions are strongly *asymmetric* in practice.
The idea was discarded on the evidence.

**Capping clause length** worked, because the model's failures concentrate in long
clauses. The sweep below quantifies it.

In [17]:
# The three defects we planted, identified by the language they conflict over.
PROBES = {"payment_terms":      ("thirty (30) days",      "fifteen (15) days"),
          "governing_law":      ("Republic of Singapore", "State of Delaware"),
          "termination_notice": ("sixty (60) days",       "ninety (90) days")}

def planted_found(results):
    hits = []
    for pid, (na, nb) in PROBES.items():
        for r in results:
            blob = (r["clause_a"] + " || " + r["clause_b"]).lower()
            if na.lower() in blob and nb.lower() in blob:
                hits.append(pid); break
    return hits

# Precision/recall sweep: re-judge at several settings and count both
# how many findings a reviewer would face and how many real defects survive.
sweep = []
for cap in (250, 350, 500, 100000):
    res, _ = detect_contradictions("contract", min_score=0.0, max_clause_chars=cap)
    for thr in (0.95, 0.99):
        flagged = [r for r in res if r["contradiction"] >= thr]
        hits = planted_found(flagged)
        sweep.append({"clause cap": "none" if cap > 9999 else cap,
                      "threshold": thr,
                      "findings to review": len(flagged),
                      "planted defects found": f"{len(hits)}/3"})
display(pd.DataFrame(sweep))
print("\nDefault: clause cap 250, threshold 0.95 -> a reviewable shortlist.")
print("Raising the cap recovers the third defect at a steep cost in precision.")
print("The trade-off is real, so it is exposed as a parameter rather than hidden.")

,clause cap,threshold,findings to review,planted defects found
0,250,0.95,9,2/3
1,250,0.99,6,1/3
2,350,0.95,17,2/3
3,350,0.99,10,1/3
4,500,0.95,34,2/3
5,500,0.99,20,1/3
6,none,0.95,102,3/3
7,none,0.99,68,2/3



Default: clause cap 250, threshold 0.95 -> a reviewable shortlist.
Raising the cap recovers the third defect at a steep cost in precision.
The trade-off is real, so it is exposed as a parameter rather than hidden.


### Did it find the planted defects?

The corpus contains three known contradictions, two of which conflict with the **original
contract text** rather than merely with each other. This is the check that matters —
anything else is a demo.

In [18]:
print(f"{'planted defect':22s} {'detected':>9s} {'score':>7s}   clauses")
print("-" * 62)
for pid, (na, nb) in PROBES.items():
    hit = None
    for r in CONTRA:
        blob = (r["clause_a"] + " || " + r["clause_b"]).lower()
        if na.lower() in blob and nb.lower() in blob:
            hit = r; break
    if hit:
        print(f"{pid:22s} {'YES':>9s} {hit['contradiction']:7.2f}   "
              f"[{hit['a_ref']}] vs [{hit['b_ref']}]")
    else:
        print(f"{pid:22s} {'MISSED':>9s} {'-':>7s}   -")

longest = max(len(c["text"]) for c in CLAUSES["contract"]
              if "fifteen (15) days" in c["text"]) if any(
              "fifteen (15) days" in c["text"] for c in CLAUSES["contract"]) else 0
print(f"""
Why payment_terms is missed at the default setting
--------------------------------------------------
The conflicting source language ("within fifteen (15) days") sits inside a
{longest}-character clause, well over the 250-character cap. That cap exists because
NLI accuracy collapses on long legal prose - so the same setting that makes the
output reviewable is what loses this defect.

This is a real recall failure, not a presentational one. Raising max_clause_chars
recovers it and returns roughly 86 findings instead of 10. Which operating point is
correct depends on whether a human is reviewing every hit or triaging a shortlist.
""")

planted defect          detected   score   clauses
--------------------------------------------------------------
payment_terms             MISSED       -   -
governing_law                YES    1.00   [10.1] vs [10.2]
termination_notice           YES    0.97   [11.1] vs [11.4]

Why payment_terms is missed at the default setting
--------------------------------------------------
The conflicting source language ("within fifteen (15) days") sits inside a
1145-character clause, well over the 250-character cap. That cap exists because
NLI accuracy collapses on long legal prose - so the same setting that makes the
output reviewable is what loses this defect.

This is a real recall failure, not a presentational one. Raising max_clause_chars
recovers it and returns roughly 86 findings instead of 10. Which operating point is
correct depends on whether a human is reviewing every hit or triaging a shortlist.



---
## Conclusion

Five features. One encoder family. No generative model at any point.

### What the architecture buys

Every finding is checkable. A PII hit is a character range you can highlight in the
original. A Q&A answer is a span with a confidence score and the clause it came from. A
contradiction is a pair of clause references and a probability. Nothing was composed, so
nothing can be confabulated.

### What it costs — stated plainly

| Limitation | Consequence |
|---|---|
| Extractive QA returns only spans that exist | cannot answer "summarise the risks" |
| 512-token window | every feature needs chunking or retrieval |
| Zero-shot NLI is not domain-tuned | thresholds are heuristic, not calibrated |
| Contradiction filtering is a heuristic | a pair below the similarity floor is never judged |

That last one is a genuine recall/compute trade-off, not an oversight: lowering
`min_sim` finds more but costs quadratically more cross-encoder passes.

### Where this would go next

Fine-tuning the NLI head on contract-specific pairs would replace the heuristic
thresholds with calibrated ones, and a playbook-deviation head (embedding each clause
against a library of approved clauses) would extend this from *finding defects* to
*proposing the approved alternative*.